# Smartphone Addiction Analysis

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.experimental import enable_iterative_imputerfrom sklearn.impute import IterativeImputerfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import RobustScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_scorefrom sklearn.cluster import KMeansfrom sklearn.metrics import silhouette_scoreimport xgboost as xgbimport lightgbm as lgbimport warningswarnings.filterwarnings('ignore')%matplotlib inline

---

## Phase 1: Load Data

In [ ]:
df = pd.read_csv('data/Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv')print(f'Shape: {df.shape}')df.head()

---

## Phase 2: Data Preprocessing

In [ ]:
df = df.drop(columns=['transaction_id', 'user_id'])print('Dropped: transaction_id, user_id')

In [ ]:
print(f'Missing before: {df.isnull().sum().sum()}')num_cols = df.select_dtypes(include=[np.number]).columns.tolist()df[num_cols] = IterativeImputer(max_iter=10, random_state=42).fit_transform(df[num_cols])cat_cols = df.select_dtypes(include=['object']).columns.tolist()for col in cat_cols:    df[col] = df[col].fillna(df[col].mode()[0])print(f'Missing after: {df.isnull().sum().sum()}')

In [ ]:
dups = df.duplicated().sum()print(f'Duplicates: {dups}')df = df.drop_duplicates()

In [ ]:
def winsorize(col, lower=0.01, upper=0.99):    return col.clip(col.quantile(lower), col.quantile(upper))for col in ['notifications_per_day', 'app_opens_per_day', 'daily_screen_time_hours']:    df[col] = winsorize(df[col])print(f'Final Shape: {df.shape}')

---

## Phase 3: Exploratory Data Analysis

In [ ]:
print('Addiction Level Distribution:')print(df['addiction_level'].value_counts())print('Addicted Label Distribution:')print(df['addicted_label'].value_counts())

In [ ]:
df.describe()

---

## Phase 4: Feature Engineering

In [ ]:
df['gaming_ratio'] = df['gaming_hours'] / df['daily_screen_time_hours']df['social_ratio'] = df['social_media_hours'] / df['daily_screen_time_hours']df['weekend_vs_weekday_ratio'] = df['weekend_screen_time'] / df['daily_screen_time_hours']print('Created: gaming_ratio, social_ratio, weekend_vs_weekday_ratio')

In [ ]:
stress_map = {'Low': 0, 'Medium': 1, 'High': 2}df['stress_level_enc'] = df['stress_level'].map(stress_map)df['academic_work_impact_enc'] = df['academic_work_impact'].map({'No': 0, 'Yes': 1})addiction_map = {'Mild': 0, 'Moderate': 1, 'Severe': 2}df['addiction_level_enc'] = df['addiction_level'].map(addiction_map)df = pd.get_dummies(df, columns=['gender'], drop_first=True)print('Encoded: stress_level, academic_work_impact, addiction_level, gender')

In [ ]:
scaler = RobustScaler()num_features = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',                'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day',                'weekend_screen_time', 'gaming_ratio', 'social_ratio', 'weekend_vs_weekday_ratio']df[num_features] = scaler.fit_transform(df[num_features])print('Scaled features with RobustScaler')

---

## Phase 5: Predictive Modeling

In [ ]:
features = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',            'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day',            'weekend_screen_time', 'stress_level_enc', 'academic_work_impact_enc',            'gaming_ratio', 'social_ratio', 'weekend_vs_weekday_ratio',            'gender_Male']X = df[features]y = df['addiction_level_enc']X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
models = {    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='mlogloss', verbosity=0),    'LightGBM': lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)}results = {}for name, model in models.items():    model.fit(X_train, y_train)    y_pred = model.predict(X_test)    results[name] = {        'accuracy': accuracy_score(y_test, y_pred),        'f1_macro': f1_score(y_test, y_pred, average='macro'),        'precision': precision_score(y_test, y_pred, average='macro'),        'recall': recall_score(y_test, y_pred, average='macro')    }    print(f'{name}: Accuracy={results[name]["accuracy"]:.4f}, F1={results[name]["f1_macro"]:.4f}')

---

## Phase 6: Model Evaluation

In [ ]:
best_model_name = max(results, key=lambda x: results[x]['f1_macro'])print(f'Best Model: {best_model_name}')best_model = models[best_model_name]

In [ ]:
y_pred = best_model.predict(X_test)print(classification_report(y_test, y_pred, target_names=['Mild', 'Moderate', 'Severe']))

In [ ]:
plt.figure(figsize=(8, 6))sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',            xticklabels=['Mild', 'Moderate', 'Severe'],            yticklabels=['Mild', 'Moderate', 'Severe'])plt.title(f'Confusion Matrix - {best_model_name}')plt.xlabel('Predicted')plt.ylabel('Actual')plt.show()

---

## Phase 7: Clustering

In [ ]:
cluster_features = ['daily_screen_time_hours', 'gaming_hours', 'social_media_hours',                    'sleep_hours', 'notifications_per_day']X_cluster = df[cluster_features]X_cluster_scaled = RobustScaler().fit_transform(X_cluster)

In [ ]:
k_range = range(2, 8)inertias = []silhouettes = []for k in k_range:    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)    kmeans.fit(X_cluster_scaled)    inertias.append(kmeans.inertia_)    silhouettes.append(silhouette_score(X_cluster_scaled, kmeans.labels_))fig, axes = plt.subplots(1, 2, figsize=(12, 4))axes[0].plot(k_range, inertias, 'bo-')axes[0].set_title('Elbow Method')axes[0].set_xlabel('k')axes[1].plot(k_range, silhouettes, 'go-')axes[1].set_title('Silhouette Score')axes[1].set_xlabel('k')plt.tight_layout()plt.show()

In [ ]:
optimal_k = list(k_range)[np.argmax(silhouettes)]print(f'Optimal k: {optimal_k}')kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)df['cluster'] = kmeans.fit_predict(X_cluster_scaled)print('Cluster Profiles:')df.groupby('cluster')[cluster_features].mean().round(2)

---

## Phase 8: Summary

In [ ]:
print('='*50)print('PHASE SUMMARY')print('='*50)print(f'Data Shape: {df.shape}')print(f'Best Model: {best_model_name}')print(f'Accuracy: {results[best_model_name]["accuracy"]:.4f}')print(f'F1 Score (Macro): {results[best_model_name]["f1_macro"]:.4f}')print(f'Optimal Clusters: {optimal_k}')print('='*50)